In [91]:
import numpy as np
import pandas as pd
from scipy import stats

## Problem 1.1

In [92]:
def ehw(X: np.ndarray, y: np.ndarray, b: np.ndarray, hc1: bool = False) -> np.ndarray[float]:
    n, p = X.shape
    B = np.linalg.inv(X.T @ X)

    resid = (y - X @ b)
    M = (X * resid[:, None]**2).T @ X

    correction = n / (n - p) if hc1 else 1.0

    return B @ M @ B * correction

In [93]:
lalonde = pd.read_csv("data/lalonde_nsw.csv")

lalonde.head(3)

,treat,age,education,black,hispanic,married,nodegree,re74,re75,re78
0,1,37,11,1,0,1,1,0.0,0.0,9930.045898
1,1,22,9,0,1,0,1,0.0,0.0,3595.894043
2,1,30,12,1,0,0,0,0.0,0.0,24909.449219


### Regression 1: Difference in Means Estimator $\hat{\tau}$

$$ y_i \approx \alpha^{DM} + \tau^{DM} w_i $$

In [94]:
y = lalonde['re78'].values
w = lalonde['treat'].values

X_dm = np.column_stack([np.ones(len(w)), w])

In [95]:
coefs_dm, _, _, _ = np.linalg.lstsq(X_dm, y, rcond=None)
tau_hat_dm = coefs_dm[1]

In [96]:
ehw_hat_dm    = ehw(X_dm, y, coefs_dm)
tau_hat_dm_se = np.sqrt(ehw_hat_dm[1, 1])

### Regression 2: Covariate Adjusted Difference in Means Estimator $\tau^{adj}$

$$ y_i \approx \alpha^{adj} + w_i \tau^{adj} + x_i^{\mathsf{T}} \beta^{adj}$$

In [97]:
y = lalonde['re78'].values
covs = [c for c in lalonde.columns if c not in ['re78', 'treat']]
X_adj = np.column_stack([np.ones(len(lalonde)), lalonde['treat'], lalonde[covs]])

X_adj.shape

(445, 10)

In [98]:
coefs_adj, _, _, _ = np.linalg.lstsq(X_adj, y, rcond=None)
tau_hat_adj = coefs_adj[1]

In [99]:
ehw_hat_adj    = ehw(X_adj, y, coefs_adj)
tau_hat_adj_se = np.sqrt(ehw_hat_adj[1, 1])

ehw_hat_adj.shape

(10, 10)

In [100]:
# Reporting results

pd.DataFrame(
    data    = [(tau_hat_dm, tau_hat_dm_se), (tau_hat_adj, tau_hat_adj_se)],
    columns = ['tau', 'se (HC0)'],
    index   = ['dm', 'adj']
).round(4)

,tau,se (HC0)
dm,1794.3424,669.3153
adj,1676.3426,669.0867


## Problem 1.2

In [101]:
tau_hat_dm_z = (abs(tau_hat_dm) - 0) / tau_hat_dm_se
tau_hat_dm_p = 2 * stats.norm.cdf(-tau_hat_dm_z)

tau_hat_adj_z = (abs(tau_hat_adj) - 0) / tau_hat_adj_se
tau_hat_adj_p = 2 * stats.norm.cdf(-tau_hat_adj_z)

In [102]:
# Reporting results

pd.DataFrame(
    data    = [(tau_hat_dm_p, tau_hat_adj_p)],
    columns = ['tau_hat_dm', 'tau_hat_adj'],
    index   = ['p-value'],
).round(4)

,tau_hat_dm,tau_hat_adj
p-value,0.0073,0.0122


## Problem 1.3

In [103]:
B: int = 1_000
SEED: int = 42

In [104]:
def permutation_test(X: np.ndarray, y: np.ndarray, B: int = 1_000, seed: int = 42) -> float:
    """
    w_i must be in the second column

    Returns:
        p-value for coefficient of w_i
    """
    rng = np.random.default_rng(seed)

    w = X[:, 1].copy()
    X_perm = X.copy()

    tau_hat = np.abs(np.linalg.lstsq(X, y, rcond=None)[0][1])

    tau_hat_reps = np.empty(B)
    for b in range(0, B):
        X_perm[:, 1]    = rng.permutation(w)
        tau_hat_reps[b] = np.linalg.lstsq(X_perm, y, rcond=None)[0][1]

    p = (1 + (np.sum(np.abs(tau_hat_reps) >= tau_hat))) / (1 + B)

    return p

In [105]:
"""
Note:
    We have equivalently seeded DM and adj, so the sequence of permutations will be the same, which works well when comparing p-values.
"""

tau_hat_dm_p_perm  = permutation_test(X_dm, y, B, SEED)
tau_hat_adj_p_perm = permutation_test(X_adj, y, B, SEED)

In [106]:
# Report results

pd.DataFrame(
    data    = [(tau_hat_dm_p_perm, tau_hat_adj_p_perm)],
    columns = ['tau_hat_dm', 'tau_hat_adj'],
    index   = ['p-value (perm test)'],
).round(4)

,tau_hat_dm,tau_hat_adj
p-value (perm test),0.007,0.013


## Problem 3.B.1

In [107]:
pulp = pd.read_csv("data/pulp.csv")

pulp

,bright,operator
0,59.8,a
1,60.0,a
2,60.8,a
3,60.8,a
4,59.8,a
5,59.8,b
6,60.2,b
7,60.4,b
8,59.9,b
9,60.0,b


In [108]:
pulp['operator'] = pulp['operator'].astype(str)
pulp['bright']   = pulp['bright'].astype(float)

In [109]:
y = pulp['bright'].to_numpy()
b = pulp['operator'].to_numpy()

In [110]:
def compute_alpha(y: np.ndarray, b: np.ndarray, cat: str = 'a') -> float:
    mask = b == cat
    return 1 / 5 * np.sum(y[mask])

In [111]:
alpha_hat_1 = compute_alpha(y, b, cat = 'a')
alpha_hat_2 = compute_alpha(y, b, cat = 'b')
alpha_hat_3 = compute_alpha(y, b, cat = 'c')
alpha_hat_4 = compute_alpha(y, b, cat = 'd')

In [112]:
print(f"alpha hat 1: {alpha_hat_1}\nalpha hat 2: {alpha_hat_2}\nalpha hat 3: {alpha_hat_3}\nalpha hat 4: {alpha_hat_4}\n")

alpha hat 1: 60.24
alpha hat 2: 60.06
alpha hat 3: 60.620000000000005
alpha hat 4: 60.68



## Problem 4.3 

In [113]:
np.set_printoptions(precision=3, suppress=True, linewidth=200)  # Pretty print our outputs!

In [114]:
D_1 = np.array([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [1, 0, 0, 1]
])

D_1 = np.repeat(D_1, 3, axis=0)

D_1     # Design matrix 1

array([[1, 0, 0, 0],
       [1, 0, 0, 0],
       [1, 0, 0, 0],
       [1, 1, 0, 0],
       [1, 1, 0, 0],
       [1, 1, 0, 0],
       [1, 0, 1, 0],
       [1, 0, 1, 0],
       [1, 0, 1, 0],
       [1, 0, 0, 1],
       [1, 0, 0, 1],
       [1, 0, 0, 1]])

In [115]:
print(np.linalg.inv(D_1.T @ D_1) @ D_1.T)

[[ 0.333  0.333  0.333  0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [-0.333 -0.333 -0.333  0.333  0.333  0.333  0.     0.     0.     0.     0.     0.   ]
 [-0.333 -0.333 -0.333  0.     0.     0.     0.333  0.333  0.333  0.     0.     0.   ]
 [-0.333 -0.333 -0.333  0.     0.     0.     0.     0.     0.     0.333  0.333  0.333]]


In [116]:
D_2 = np.identity(4)

D_2 = np.repeat(D_2, 3, axis=0)

D_2     # Design matrix 2

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.]])

In [117]:
print(np.linalg.inv(D_2.T @ D_2) @ D_2.T)

[[0.333 0.333 0.333 0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.333 0.333 0.333 0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.333 0.333 0.333 0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.333 0.333 0.333]]


In [118]:
D_3 = np.array([
    [1, 0, 0, 0],
    [1, 1, 1, 0],
    [1, 1, 0, 1],
    [1, 1, -1, -1]
])

D_3 = np.repeat(D_3, 3, axis=0)

D_3      # Design matrix 3

array([[ 1,  0,  0,  0],
       [ 1,  0,  0,  0],
       [ 1,  0,  0,  0],
       [ 1,  1,  1,  0],
       [ 1,  1,  1,  0],
       [ 1,  1,  1,  0],
       [ 1,  1,  0,  1],
       [ 1,  1,  0,  1],
       [ 1,  1,  0,  1],
       [ 1,  1, -1, -1],
       [ 1,  1, -1, -1],
       [ 1,  1, -1, -1]])

In [119]:
print(np.linalg.inv(D_3.T @ D_3) @ D_3.T)

[[ 0.333  0.333  0.333  0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [-0.333 -0.333 -0.333  0.111  0.111  0.111  0.111  0.111  0.111  0.111  0.111  0.111]
 [ 0.     0.     0.     0.222  0.222  0.222 -0.111 -0.111 -0.111 -0.111 -0.111 -0.111]
 [ 0.     0.     0.    -0.111 -0.111 -0.111  0.222  0.222  0.222 -0.111 -0.111 -0.111]]


In [120]:
print("Column Sums")
np.sum(np.linalg.inv(D_3.T @ D_3) @ D_3.T, axis = 0, keepdims=True)

Column Sums


array([[ 0.   ,  0.   ,  0.   ,  0.222,  0.222,  0.222,  0.222,  0.222,  0.222, -0.111, -0.111, -0.111]])

In [121]:
print("Row Sums")
np.sum(np.linalg.inv(D_3.T @ D_3) @ D_3.T, axis = 1, keepdims=True)

Row Sums


array([[1.],
       [0.],
       [0.],
       [0.]])

In [122]:
D_4 = np.array([
    [1, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, -1],
    [1, 1, -1, 0]
])

D_4 = np.repeat(D_4, 3, axis=0)

D_4      # Design matrix 4

array([[ 1,  1,  1,  0],
       [ 1,  1,  1,  0],
       [ 1,  1,  1,  0],
       [ 1,  0,  0,  1],
       [ 1,  0,  0,  1],
       [ 1,  0,  0,  1],
       [ 1,  0,  0, -1],
       [ 1,  0,  0, -1],
       [ 1,  0,  0, -1],
       [ 1,  1, -1,  0],
       [ 1,  1, -1,  0],
       [ 1,  1, -1,  0]])

In [123]:
print(np.linalg.inv(D_4.T @ D_4) @ D_4.T)

[[ 0.     0.     0.     0.167  0.167  0.167  0.167  0.167  0.167  0.     0.     0.   ]
 [ 0.167  0.167  0.167 -0.167 -0.167 -0.167 -0.167 -0.167 -0.167  0.167  0.167  0.167]
 [ 0.167  0.167  0.167  0.     0.     0.     0.     0.     0.    -0.167 -0.167 -0.167]
 [ 0.     0.     0.     0.167  0.167  0.167 -0.167 -0.167 -0.167  0.     0.     0.   ]]


In [124]:
print("Column Sums")
np.sum(np.linalg.inv(D_4.T @ D_4) @ D_4.T, axis = 0, keepdims=True)

Column Sums


array([[ 0.333,  0.333,  0.333,  0.167,  0.167,  0.167, -0.167, -0.167, -0.167,  0.   ,  0.   ,  0.   ]])

In [125]:
print("Row Sums")
np.sum(np.linalg.inv(D_4.T @ D_4) @ D_4.T, axis = 1, keepdims=True)

Row Sums


array([[ 1.],
       [-0.],
       [ 0.],
       [ 0.]])

### Comments

First, let's recall the normal equations $$ \hat{\mathbf{\beta}} = (X^{\mathsf{T}}X)^{-1}X^{\mathsf{T}}\mathbf{y} $$
and notice that $(X^{\top}X)^{-1}X^{\top} \in \mathbb{R}^{4 \times 12}$ is a matrix of weights being applied to our response. Concretely,
$(X^{\top}X)^{-1}X^{\top}: \mathbb{R}^{12} \longmapsto \mathbb{R}^{4}$, where our response $\mathbf{y} \in \mathbb{R}^{12}$ and our coefficient $\hat{\mathbf{\beta}} \in \mathbb{R}^{4}$. Next, we analyze each mapping.

- **Design Matrix 1:** Our mapping assigns the mean of the control response to $\hat{\beta_0}$, removes the mean of the control response from the treatment I response mean, assigning the value to $\hat{\beta_1}$, removes the mean of the control response from the treatment II response mean, assigning the value to $\hat{\beta_2}$, and removes the mean of the control response from the treatment III response mean, assigning the value to $\hat{\beta_3}$.

- **Design Matrix 2:** Our mapping assigns the mean of the control of our response to $\hat{\beta_0}$, the mean response of treatment I to $\hat{\beta_1}$, the mean response of treatment II to $\hat{\beta_2}$, and the mean response of treatment III to $\hat{\beta_3}$.

- **Design Matrix 3:** Our mapping assigns the mean of the control of our response to $\hat{\beta_0}$. The control of interest is in row two, with $-\frac{1}{3}$ weight on the control response and $\frac{1}{9}$ weight on the responses for treatments I, II, and III. Rows three and four sum to zero and represent the comparisons we made between treatment groups in our equation.

- **Design Matrix 4:** Our mapping assigns the mean response of treatments I and II to $\hat{\beta_0}$. The control of interest is represented in row two, with $\frac{1}{6}$ weights applied to the control response and treatment III response, and $-\frac{1}{6}$ weights applied to the responses for treatments I and II. Rows three and four sum to zero and represent the comparisons we made between treatment groups in our equation.

## Problem 4.4

In [ ]:
# Original design matrix from pg 25 of the reading
D = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0, 1],
    [0, 0, 0, 0, 0, 1, 0, 1]
])

# Add indicator for "Y" treatment group
D = np.column_stack([D, np.concatenate([np.zeros(6), np.ones(6)])])

# Design matrix with new "Y" treatment group indicator
D

array([[1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 1., 0., 0., 0., 1., 1.],
       [0., 0., 0., 0., 1., 0., 0., 1., 1.],
       [0., 0., 0., 0., 0., 1., 0., 1., 1.]])

In [132]:
# expression as our response
y = np.array([1.01, 1.04, 1.04, 1.99, 2.36, 2.00, 2.89, 3.12, 2.98, 5.00, 4.92, 4.78])

y

array([1.01, 1.04, 1.04, 1.99, 2.36, 2.  , 2.89, 3.12, 2.98, 5.  , 4.92, 4.78])

In [131]:
A = D.T @ D
b = D.T @ y

try:
    coefs = np.linalg.solve(A, b)
except np.linalg.LinAlgError as e:
    print(f"Error computing OLS for given design matrix: {e}")

Error computing OLS for given design matrix: Singular matrix


In [140]:
np.sum(D[:, 3:6], axis=1) == D[:, -1]

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True])

In [145]:
print(f"Design Matrix\n  Rank: {np.linalg.matrix_rank(D)}\n  Cols: {D.shape[1]}")

Design Matrix
  Rank: 8
  Cols: 9


### Comments

Notice that a linear combination of columns four, five, and six is equivalent to column nine (our 1 if treatment == "Y" and 0 otherwise column). So, we can see that we have already encoded this information in our matrix, and our "new feature" is linearly dependent on the information we have already included in our matrix. Therefore, we cannot estimate the additive treatment effect from this design matrix.

## Scratch & Extra Work (can ignore)

In [ ]:
alpha = np.array([compute_alpha(y, b, cat = c) for c in pulp['operator'].unique()])

In [ ]:
np.mean(y)

60.39999999999999

In [ ]:
np.mean(alpha)

60.400000000000006

In [ ]:
B = np.array([
    [ 1, 0, 0, 0],
    [-1, 1, 0, 0],
    [-1, 0, 1, 0],
    [-1, 0, 0, 1]
])
beta = B @ alpha
beta

array([60.24, -0.18,  0.38,  0.44])

In [ ]:
G = 1/4 * np.array([
    [ 1, 1, 1, 1],
    [ 3,-1,-1,-1],
    [-1, 3,-1,-1],
    [-1,-1, 3,-1]
])
gamma = G @ alpha
gamma

array([60.4 , -0.16, -0.34,  0.22])

In [ ]:
print(f"B Invertible: {np.round(np.linalg.det(B), 10) != 0}\nG Invertible: {np.round(np.linalg.det(G), 10) != 0}")

B Invertible: True
G Invertible: True


In [ ]:
from typing import Tuple

In [ ]:
def the_WILD_bootstrap(X: np.ndarray, y: np.ndarray, alpha: float = 0.05, B: int = 1_000, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    n, p = X.shape

    rng = np.random.default_rng(seed = seed)

    A = (X.T @ X)
    b = (X.T @ y)
    beta_hat = np.linalg.solve(A, b)

    residuals = y - X @ beta_hat

    replicants = []
    for _ in range(B):
        u = rng.normal(0, 1, size = n)
        y_b = X @ beta_hat + np.multiply(residuals, u)
        b_b = (X.T @ y_b)

        replicants.append(np.linalg.solve(A, b_b))

    return np.quantile(replicants, q = alpha / 2, axis = 0), np.quantile(replicants, q = 1 - alpha / 2, axis = 0)   